## Imports

In [1]:
import os
import pandas as pd
from pathlib import Path
import pydicom
import pydicom_seg as dcmseg
import numpy as np
import SimpleITK as sitk
#from utils_pyradiomics import create_path_df
import matplotlib.pyplot as plt

## Loading dicom files

In [2]:
def read_scans(ct_path, seg_path):

    seg = pydicom.dcmread(list(seg_path.glob('*.dcm'))[0])
    result_seg = seg_reader.read(seg)
    dcm_paths = sorted(ct_path.glob('*.dcm'))
    dcm_files = ser_reader.GetGDCMSeriesFileNames(str(ct_path))
    ser_reader.SetFileNames(dcm_files)
    dcms = ser_reader.Execute()

    return dcms, result_seg, seg

def get_neoplasm_segment_image(result_seg):
    for seg_num, info in result_seg.segment_infos.items():
        if 'Neoplasm' in info.get('SegmentLabel', ''):
            return result_seg.segment_image(seg_num)
    raise ValueError('Neoplasm segment not found')

In [3]:
def create_path_df(general_dir):
    
    path_records = []

    for patient_dir in general_dir.iterdir():
        if not patient_dir.is_dir():
            continue

        scan_id = patient_dir.name
        
        for study_dir in patient_dir.iterdir():
            if not study_dir.is_dir():
                continue

            for series_dir in study_dir.iterdir():
                if not series_dir.is_dir():
                    continue

                #select whether ct scan series or segmentation based on name/length
                if 'Segmentation' in series_dir.name and any(series_dir.glob('*.dcm')):
                    seg_series = series_dir
                    continue

                if any(series_dir.glob('*.dcm')) and len(list(series_dir.glob('*.dcm'))) >= 10:
                    ct_series = series_dir

            if ct_series is not None and seg_series is not None:
                path_records.append({
                    'scan_id': scan_id,
                    'path_ct': ct_series,
                    'path_mask':seg_series
                })
            else:
                print(f"No valid paths for {patient_dir.name}/{study_dir.name}: ct_series={ct_series is not None}, seg_series={seg_series is not None}")

    path_df = pd.DataFrame(path_records, columns=['scan_id', 'path_ct', 'path_mask'])

    return path_df

In [4]:
general_dir = Path(os.path.expanduser('~/project/xAI-in-NSCLC/NSCLC-Radiomics'))

path_df = create_path_df(general_dir)



In [5]:
path_df = path_df[:30]

In [6]:
#Edit this so that you have a toggle for fixing the segmentation

is_fix_segmentation = bool(True)

In [11]:
seg_reader = dcmseg.SegmentReader()
ser_reader = sitk.ImageSeriesReader()

scan_data = {}
for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']

    sitk_dcms, result_seg, seg = read_scans(ct_path, mask_path)
    neoplasm_segment_img = get_neoplasm_segment_image(result_seg)#


    if is_fix_segmentation == True:
        neoplasm_segment_img = fix_seg(neoplasm_segment_img, sitk_dcms)

    spacing = sitk_dcms.GetSpacing()
    print('first, seg')
    print("Spacing (x, y, z):", neoplasm_segment_img.GetSpacing())
    print("Spacing (x, y, z):", spacing)

first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.977, 0.977, 3.0)
Spacing (x, y, z): (0.977, 0.977, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
first, seg
Spacing (x, y, z): (0.9765625, 0.9765625, 3.0)
Spacing (x, y

In [7]:
def fix_seg(seg_img, ct_imgs):
    fixed_seg = sitk.Cast(seg_img, sitk.sitkUInt8)
    return fixed_seg

In [14]:
seg_reader = dcmseg.SegmentReader()
ser_reader = sitk.ImageSeriesReader()


scan_data = {}
for _, row in path_df.iterrows():
    scan_id = row['scan_id']
    ct_path = row['path_ct']
    mask_path = row['path_mask']

    sitk_dcms, result_seg, seg = read_scans(ct_path, mask_path)
    neoplasm_segment_img = get_neoplasm_segment_image(result_seg)

    if is_fix_segmentation == True:
        neoplasm_segment_img = fix_seg(neoplasm_segment_img, sitk_dcms)

    ct_array = sitk.GetArrayFromImage(sitk_dcms)
    seg_array = sitk.GetArrayFromImage(neoplasm_segment_img)
    slice_indices = list(np.where(seg_array.sum(axis=(1, 2)) > 0)[0])

    scan_data[scan_id] = {
        'ct_array': ct_array,
        'seg_array': seg_array,
        'slice_indices': slice_indices,
    }

In [15]:
def show_overlay(scan_id, slice_idx, alpha=0.5):
    data = scan_data[scan_id]
    ct_array = data['ct_array']
    seg_array = data['seg_array']

    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Display CT scan
    ax.imshow(ct_array[slice_idx], cmap='gray')
    
    # Overlay segmentation with transparency
    ax.imshow(seg_array[slice_idx], cmap='Reds', alpha=alpha)
    ax.set_title(f'{scan_id} slice {slice_idx}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

In [16]:
import ipywidgets as widgets
from IPython.display import display

In [17]:
# Create widgets for scan and mismatched slice selection
scan_selector = widgets.Dropdown(
    options=list(scan_data.keys()),
    description='Scan:',
    value=list(scan_data.keys())[0],
)

initial_slices = scan_data[scan_selector.value]['slice_indices']
if not initial_slices:
    raise RuntimeError(f'No segmented slices found for scan {scan_selector.value}')

slice_selector = widgets.SelectionSlider(
    options=initial_slices,
    description='Slice:',
    value=initial_slices[0],
    continuous_update=False,
)

alpha_selector = widgets.FloatSlider(
    min=0.0,
    max=1.0,
    step=0.1,
    value=0.5,
    description='Alpha:',
)


def update_slice_options(change):
    new_scan = change['new']
    new_slices = scan_data[new_scan]['slice_indices']
    slice_selector.options = new_slices
    slice_selector.value = new_slices[0] if new_slices else None

scan_selector.observe(update_slice_options, names='value')


def show_overlay(scan_id, slice_idx, alpha=0.5):
    data = scan_data[scan_id]
    ct_array = data['ct_array']
    seg_array = data['seg_array']

    fig, ax = plt.subplots(figsize=(8, 8))
    
    # Display CT scan
    ax.imshow(ct_array[slice_idx], cmap='gray')
    
    # Overlay segmentation with transparency
    ax.imshow(seg_array[slice_idx], cmap='Reds', alpha=alpha)
    ax.set_title(f'{scan_id} slice {slice_idx}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    plt.close()

output = widgets.interactive_output(
    show_overlay,
    {
        'scan_id': scan_selector,
        'slice_idx': slice_selector,
        'alpha': alpha_selector,
    }
)

display(widgets.VBox([
    scan_selector,
    slice_selector,
    alpha_selector,
    output,
]))